In [1]:
import pandas as pd
import os

all_data = []

# loop through uploaded files
for file in os.listdir():
    if file.endswith(".psv"):
        df = pd.read_csv(file, sep='|')
        df["patient_id"] = file
        all_data.append(df)

# combine all
final_df = pd.concat(all_data, ignore_index=True)

# save file
final_df.to_csv("combined_data.csv", index=False)

print("Done! Shape:", final_df.shape)

Done! Shape: (2033, 42)


In [2]:
print(final_df.head())
print(final_df.columns)

     HR  O2Sat   Temp   SBP    MAP   DBP  Resp  EtCO2  BaseExcess  HCO3  ...  \
0   NaN    NaN    NaN   NaN    NaN   NaN   NaN    NaN         NaN   NaN  ...   
1  64.0   97.0    NaN  84.0  58.00   NaN  15.0    NaN         NaN   NaN  ...   
2  60.0   98.0  36.67  92.0  60.67   NaN  14.0    NaN         NaN   NaN  ...   
3  64.0    NaN    NaN  98.5  55.00  66.0  19.0    NaN         NaN   NaN  ...   
4  61.0    NaN    NaN  90.0  63.00  52.0  12.0    NaN         NaN   NaN  ...   

   Fibrinogen  Platelets   Age  Gender  Unit1  Unit2  HospAdmTime  ICULOS  \
0         NaN        NaN  49.8       0    1.0    0.0       -25.96       1   
1         NaN        NaN  49.8       0    1.0    0.0       -25.96       2   
2         NaN        NaN  49.8       0    1.0    0.0       -25.96       3   
3         NaN        NaN  49.8       0    1.0    0.0       -25.96       4   
4         NaN        NaN  49.8       0    1.0    0.0       -25.96       5   

   SepsisLabel   patient_id  
0            0  p000045.ps

In [3]:
# check missing values
print(final_df.isnull().sum())

HR                   180
O2Sat                299
Temp                1330
SBP                  412
MAP                  236
DBP                 1065
Resp                 240
EtCO2               2033
BaseExcess          1806
HCO3                1859
FiO2                1762
pH                  1798
PaCO2               1840
SaO2                1945
AST                 2008
BUN                 1856
Alkalinephos        2009
Calcium             1930
Chloride            1858
Creatinine          1883
Bilirubin_direct    2033
Glucose             1807
Lactate             1950
Magnesium           1883
Phosphate           1927
Potassium           1811
Bilirubin_total     2014
TroponinI           2031
Hct                 1821
Hgb                 1859
PTT                 1950
WBC                 1878
Fibrinogen          2024
Platelets           1900
Age                    0
Gender                 0
Unit1               1025
Unit2               1025
HospAdmTime            0
ICULOS                 0


In [4]:
features = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'Age']
target = 'SepsisLabel'

df = final_df[features + [target]]

In [5]:
df = df.fillna(df.mean())

In [6]:
from sklearn.model_selection import train_test_split

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
from sklearn.model_selection import train_test_split

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)


RandomForestClassifier()

In [9]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.99      0.98       391
           1       0.50      0.12      0.20        16

    accuracy                           0.96       407
   macro avg       0.73      0.56      0.59       407
weighted avg       0.95      0.96      0.95       407



In [10]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[389   2]
 [ 14   2]]


In [11]:
from sklearn.utils import resample

# combine X and y
df_combined = pd.concat([X_train, y_train], axis=1)

# separate classes
df_majority = df_combined[df_combined.SepsisLabel == 0]
df_minority = df_combined[df_combined.SepsisLabel == 1]

# upsample minority
df_minority_upsampled = resample(df_minority,
                                replace=True,
                                n_samples=len(df_majority),
                                random_state=42)

# combine back
df_balanced = pd.concat([df_majority, df_minority_upsampled])

# split again
X_train_bal = df_balanced[features]
y_train_bal = df_balanced['SepsisLabel']

In [12]:
model = RandomForestClassifier()
model.fit(X_train_bal, y_train_bal)

RandomForestClassifier()

In [13]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.97      0.98       391
           1       0.50      0.62      0.56        16

    accuracy                           0.96       407
   macro avg       0.74      0.80      0.77       407
weighted avg       0.97      0.96      0.96       407



In [14]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.98      0.97      0.98       391
           1       0.50      0.62      0.56        16

    accuracy                           0.96       407
   macro avg       0.74      0.80      0.77       407
weighted avg       0.97      0.96      0.96       407



In [15]:
important_cols = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'Age', 'Gender', 'ICULOS', 'SepsisLabel']

df = final_df[important_cols].copy()

print(df.head())
print(df.shape)


     HR  O2Sat   Temp   SBP    MAP   DBP  Resp   Age  Gender  ICULOS  \
0   NaN    NaN    NaN   NaN    NaN   NaN   NaN  49.8       0       1   
1  64.0   97.0    NaN  84.0  58.00   NaN  15.0  49.8       0       2   
2  60.0   98.0  36.67  92.0  60.67   NaN  14.0  49.8       0       3   
3  64.0    NaN    NaN  98.5  55.00  66.0  19.0  49.8       0       4   
4  61.0    NaN    NaN  90.0  63.00  52.0  12.0  49.8       0       5   

   SepsisLabel  
0            0  
1            0  
2            0  
3            0  
4            0  
(2033, 11)


In [16]:
print(df.isnull().sum())

HR              180
O2Sat           299
Temp           1330
SBP             412
MAP             236
DBP            1065
Resp            240
Age               0
Gender            0
ICULOS            0
SepsisLabel       0
dtype: int64


In [17]:
df_with_pid = final_df[['patient_id'] + important_cols].copy()

df_with_pid[important_cols[:-1]] = (
    df_with_pid.groupby('patient_id')[important_cols[:-1]]
    .apply(lambda group: group.ffill().bfill())
    .reset_index(level=0, drop=True)
)

for col in important_cols[:-1]:
    df_with_pid[col] = df_with_pid[col].fillna(df_with_pid[col].median())

df = df_with_pid[important_cols].copy()

print(df.isnull().sum())

HR             0
O2Sat          0
Temp           0
SBP            0
MAP            0
DBP            0
Resp           0
Age            0
Gender         0
ICULOS         0
SepsisLabel    0
dtype: int64


In [18]:
print(df['SepsisLabel'].value_counts())
print(df['SepsisLabel'].value_counts(normalize=True))

SepsisLabel
0    1954
1      79
Name: count, dtype: int64
SepsisLabel
0    0.961141
1    0.038859
Name: proportion, dtype: float64


In [19]:
X = df.drop('SepsisLabel', axis=1)
y = df['SepsisLabel']

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(1626, 10) (407, 10)


In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [22]:
df.to_csv("cleaned_sepsis_data.csv", index=False)
print("Cleaned dataset saved.")

Cleaned dataset saved.


In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

lr_model = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression Results")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1-score:", f1_score(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))

Logistic Regression Results
Accuracy: 0.7002457002457002
Precision: 0.11029411764705882
Recall: 0.9375
F1-score: 0.19736842105263158
ROC-AUC: 0.8462276214833759


In [24]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest Results")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1-score:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

Random Forest Results
Accuracy: 0.9828009828009828
Precision: 1.0
Recall: 0.5625
F1-score: 0.72
ROC-AUC: 0.9912084398976981


In [25]:
!pip install xgboost

In [26]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost Results")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1-score:", f1_score(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

XGBoost Results
Accuracy: 0.9901719901719902
Precision: 0.875
Recall: 0.875
F1-score: 0.875
ROC-AUC: 0.9947250639386189


In [27]:
import pandas as pd

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb)
    ],
    'F1-score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_rf),
        roc_auc_score(y_test, y_prob_xgb)
    ]
})

print(results)

                 Model  Accuracy  Precision  Recall  F1-score   ROC-AUC
0  Logistic Regression  0.700246   0.110294  0.9375  0.197368  0.846228
1        Random Forest  0.982801   1.000000  0.5625  0.720000  0.991208
2              XGBoost  0.990172   0.875000  0.8750  0.875000  0.994725


In [28]:
important_cols = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'Age', 'Gender', 'ICULOS', 'SepsisLabel']

df_with_pid = final_df[['patient_id'] + important_cols].copy()

feature_cols = [c for c in important_cols if c != 'SepsisLabel']

df_with_pid[feature_cols] = (
    df_with_pid.groupby('patient_id')[feature_cols]
    .apply(lambda group: group.ffill().bfill())
    .reset_index(level=0, drop=True)
)

for col in feature_cols:
    df_with_pid[col] = df_with_pid[col].fillna(df_with_pid[col].median())

print(df_with_pid.head())
print(df_with_pid.isnull().sum())

    patient_id    HR  O2Sat   Temp   SBP    MAP   DBP  Resp   Age  Gender  \
0  p000045.psv  64.0   97.0  36.67  84.0  58.00  66.0  15.0  49.8       0   
1  p000045.psv  64.0   97.0  36.67  84.0  58.00  66.0  15.0  49.8       0   
2  p000045.psv  60.0   98.0  36.67  92.0  60.67  66.0  14.0  49.8       0   
3  p000045.psv  64.0   98.0  36.67  98.5  55.00  66.0  19.0  49.8       0   
4  p000045.psv  61.0   98.0  36.67  90.0  63.00  52.0  12.0  49.8       0   

   ICULOS  SepsisLabel  
0       1            0  
1       2            0  
2       3            0  
3       4            0  
4       5            0  
patient_id     0
HR             0
O2Sat          0
Temp           0
SBP            0
MAP            0
DBP            0
Resp           0
Age            0
Gender         0
ICULOS         0
SepsisLabel    0
dtype: int64


In [29]:
from sklearn.model_selection import train_test_split

# unique patients
patient_ids = df_with_pid['patient_id'].unique()

# split patients
train_patients, test_patients = train_test_split(
    patient_ids,
    test_size=0.2,
    random_state=42
)

# create train and test sets using patient_id
train_df = df_with_pid[df_with_pid['patient_id'].isin(train_patients)].copy()
test_df  = df_with_pid[df_with_pid['patient_id'].isin(test_patients)].copy()

print("Train patients:", len(train_patients))
print("Test patients:", len(test_patients))
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train patients: 40
Test patients: 10
Train shape: (1589, 12)
Test shape: (444, 12)


In [30]:
features = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'Age', 'Gender', 'ICULOS']
target = 'SepsisLabel'

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

print(X_train.shape, X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())


(1589, 10) (444, 10)
SepsisLabel
0    1549
1      40
Name: count, dtype: int64
SepsisLabel
0    405
1     39
Name: count, dtype: int64


In [31]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

lr_model = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_lr, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred_lr, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))

Logistic Regression
Accuracy: 0.6666666666666666
Precision: 0.08396946564885496
Recall: 0.28205128205128205
F1-score: 0.12941176470588237
ROC-AUC: 0.4498258942703387


In [33]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_rf, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred_rf, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

Random Forest
Accuracy: 0.9099099099099099
Precision: 0.0
Recall: 0.0
F1-score: 0.0
ROC-AUC: 0.48838239949351064


In [34]:
!pip install xgboost
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_xgb, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred_xgb, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

XGBoost
Accuracy: 0.9099099099099099
Precision: 0.3333333333333333
Recall: 0.02564102564102564
F1-score: 0.047619047619047616
ROC-AUC: 0.43171889838556504


In [35]:
!pip install imbalanced-learn

In [36]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before:", y_train.value_counts())
print("After:", y_train_smote.value_counts())

Before: SepsisLabel
0    1549
1      40
Name: count, dtype: int64
After: SepsisLabel
0    1549
1    1549
Name: count, dtype: int64


In [37]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf_model.fit(X_train_smote, y_train_smote)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest (SMOTE)")
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1:", f1_score(y_test, y_pred_rf))

Random Forest (SMOTE)
Recall: 0.0
F1: 0.0


In [38]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=10,  # important
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train_smote, y_train_smote)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost (SMOTE)")
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1:", f1_score(y_test, y_pred_xgb))

XGBoost (SMOTE)
Recall: 0.02564102564102564
F1: 0.034482758620689655


In [39]:
# aggregate per patient
patient_df = df_with_pid.groupby('patient_id').agg({
    'HR': 'mean',
    'O2Sat': 'mean',
    'Temp': 'mean',
    'SBP': 'mean',
    'MAP': 'mean',
    'DBP': 'mean',
    'Resp': 'mean',
    'Age': 'first',
    'Gender': 'first',
    'ICULOS': 'max',
    'SepsisLabel': 'max'  # important
}).reset_index()

print(patient_df.head())
print(patient_df['SepsisLabel'].value_counts())

    patient_id          HR      O2Sat       Temp         SBP        MAP  \
0  p000001.psv  102.055556  91.398148  36.636296  127.444444  88.253148   
1  p000002.psv   60.956522  97.086957  36.181739  129.173913  66.630435   
2  p000003.psv   79.927083  95.333333  37.469583  139.968750  80.968542   
3  p000004.psv  102.672414  98.155172  36.447931  112.551724  66.770000   
4  p000005.psv   76.604167  97.677083  37.072292  135.072917  90.364583   

         DBP       Resp    Age  Gender  ICULOS  SepsisLabel  
0  60.000000  24.574074  83.14       0      54            0  
1  41.565217  14.608696  75.91       0      23            0  
2  54.072917  25.531250  45.82       0      48            0  
3  51.482759  18.758621  65.71       0      29            0  
4  60.000000  15.447917  28.09       1      49            0  
SepsisLabel
0    42
1     8
Name: count, dtype: int64


In [40]:
patient_df['PulsePressure'] = patient_df['SBP'] - patient_df['DBP']
patient_df['ShockIndex'] = patient_df['HR'] / patient_df['SBP']
patient_df['ShockIndex'] = patient_df['ShockIndex'].replace([float('inf'), -float('inf')], 0)
patient_df['ShockIndex'] = patient_df['ShockIndex'].fillna(0)

In [41]:
from sklearn.model_selection import train_test_split

X = patient_df.drop(['SepsisLabel', 'patient_id'], axis=1)
y = patient_df['SepsisLabel']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [42]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [43]:
rf_model = RandomForestClassifier(n_estimators=300, random_state=42)
rf_model.fit(X_train_smote, y_train_smote)

y_pred = rf_model.predict(X_test)

from sklearn.metrics import recall_score, f1_score

print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Recall: 0.5
F1: 0.5


In [44]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_prob = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.88      0.88      0.88         8
           1       0.50      0.50      0.50         2

    accuracy                           0.80        10
   macro avg       0.69      0.69      0.69        10
weighted avg       0.80      0.80      0.80        10

[[7 1]
 [1 1]]
ROC-AUC: 0.5625


In [45]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_prob = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.88      0.88      0.88         8
           1       0.50      0.50      0.50         2

    accuracy                           0.80        10
   macro avg       0.69      0.69      0.69        10
weighted avg       0.80      0.80      0.80        10

[[7 1]
 [1 1]]
ROC-AUC: 0.5625


In [51]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [54]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [55]:
!unzip /content/drive/MyDrive/training_setA.zip -d data

Archive:  /content/drive/MyDrive/training_setA.zip.zip
replace data/p000050.psv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: data/p000050.psv        
  inflating: data/p000049.psv        
  inflating: data/p000048.psv        
  inflating: data/p000047.psv        
  inflating: data/p000046.psv        
  inflating: data/p000045.psv        
  inflating: data/p000044.psv        
  inflating: data/p000043.psv        
  inflating: data/p000042.psv        
  inflating: data/p000041.psv        
  inflating: data/p000040.psv        
  inflating: data/p000039.psv        
  inflating: data/p000038.psv        
  inflating: data/p000037.psv        
  inflating: data/p000036.psv        
  inflating: data/p000035.psv        
  inflating: data/p000034.psv        
  inflating: data/p000033.psv        
  inflating: data/p000032.psv        
  inflating: data/p000031.psv        
  inflating: data/p000030.psv        
  inflating: data/p000029.psv        
  inflating: data/p000028.psv        
  inf

In [56]:
import pandas as pd
import os

folder_path = "data"

all_data = []

for file in os.listdir(folder_path):
    if file.endswith(".psv"):
        df = pd.read_csv(os.path.join(folder_path, file), sep='|')
        df["patient_id"] = file
        all_data.append(df)

final_df = pd.concat(all_data, ignore_index=True)

print("Dataset shape:", final_df.shape)

Dataset shape: (3754, 42)


In [57]:
patient_df = final_df.groupby('patient_id').agg({
    'HR': 'mean',
    'O2Sat': 'mean',
    'Temp': 'mean',
    'SBP': 'mean',
    'MAP': 'mean',
    'DBP': 'mean',
    'Resp': 'mean',
    'Age': 'first',
    'Gender': 'first',
    'ICULOS': 'max',
    'SepsisLabel': 'max'
}).reset_index()

print(patient_df.shape)
print(patient_df['SepsisLabel'].value_counts())
patient_df.head()

(101, 12)
SepsisLabel
0    87
1    14
Name: count, dtype: int64


,patient_id,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,Age,Gender,ICULOS,SepsisLabel
0,p000001.psv,101.571429,91.477273,36.778000,126.809524,87.261905,NaN,24.820000,83.14,0,54,0
1,p000002.psv,60.954545,97.000000,36.165000,136.600000,66.704545,44.066667,14.236842,75.91,0,23,0
2,p000003.psv,79.611111,95.431818,37.609375,140.033333,81.048000,54.392857,25.633333,45.82,0,48,0
3,p000004.psv,102.444444,98.203704,36.455000,113.019231,67.147308,51.428571,18.884615,65.71,0,29,0
4,p000005.psv,73.916667,97.500000,36.992222,132.770833,87.088235,NaN,16.500000,28.09,1,49,0


In [58]:
patient_df['PulsePressure'] = patient_df['SBP'] - patient_df['DBP']
patient_df['ShockIndex'] = patient_df['HR'] / patient_df['SBP']

patient_df['ShockIndex'] = patient_df['ShockIndex'].replace([float('inf'), -float('inf')], 0)
patient_df['ShockIndex'] = patient_df['ShockIndex'].fillna(0)

In [59]:
from sklearn.model_selection import train_test_split

X = patient_df.drop(['patient_id', 'SepsisLabel'], axis=1)
y = patient_df['SepsisLabel']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

(80, 12) (21, 12)
SepsisLabel
0    69
1    11
Name: count, dtype: int64
SepsisLabel
0    18
1     3
Name: count, dtype: int64


In [63]:
print(X_train.isnull().sum())
print(X_test.isnull().sum())

HR                0
O2Sat             1
Temp              1
SBP               1
MAP               0
DBP              36
Resp              0
Age               0
Gender            0
ICULOS            0
PulsePressure    36
ShockIndex        0
dtype: int64
HR               0
O2Sat            0
Temp             0
SBP              0
MAP              0
DBP              7
Resp             0
Age              0
Gender           0
ICULOS           0
PulsePressure    7
ShockIndex       0
dtype: int64


In [64]:
# fill missing values with median of training data
train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

In [65]:
print(X_train.isnull().sum())
print(X_test.isnull().sum())

HR               0
O2Sat            0
Temp             0
SBP              0
MAP              0
DBP              0
Resp             0
Age              0
Gender           0
ICULOS           0
PulsePressure    0
ShockIndex       0
dtype: int64
HR               0
O2Sat            0
Temp             0
SBP              0
MAP              0
DBP              0
Resp             0
Age              0
Gender           0
ICULOS           0
PulsePressure    0
ShockIndex       0
dtype: int64


In [66]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before:")
print(y_train.value_counts())

print("After:")
print(y_train_smote.value_counts())

Before:
SepsisLabel
0    69
1    11
Name: count, dtype: int64
After:
SepsisLabel
0    69
1    69
Name: count, dtype: int64


In [67]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf_model.fit(X_train_smote, y_train_smote)

y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

In [68]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.89      0.89      0.89        18
           1       0.33      0.33      0.33         3

    accuracy                           0.81        21
   macro avg       0.61      0.61      0.61        21
weighted avg       0.81      0.81      0.81        21

[[16  2]
 [ 2  1]]
ROC-AUC: 0.7222222222222221


In [69]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

rf_model_tuned = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

rf_model_tuned.fit(X_train_smote, y_train_smote)

y_prob_rf = rf_model_tuned.predict_proba(X_test)[:, 1]
y_pred_rf = (y_prob_rf >= 0.5).astype(int)

print("Random Forest Tuned - threshold 0.5")
print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

Random Forest Tuned - threshold 0.5
              precision    recall  f1-score   support

           0       0.89      0.89      0.89        18
           1       0.33      0.33      0.33         3

    accuracy                           0.81        21
   macro avg       0.61      0.61      0.61        21
weighted avg       0.81      0.81      0.81        21

[[16  2]
 [ 2  1]]
ROC-AUC: 0.7777777777777778


In [70]:
y_pred_rf_03 = (y_prob_rf >= 0.3).astype(int)

print("Random Forest Tuned - threshold 0.3")
print(classification_report(y_test, y_pred_rf_03))
print(confusion_matrix(y_test, y_pred_rf_03))

Random Forest Tuned - threshold 0.3
              precision    recall  f1-score   support

           0       0.93      0.72      0.81        18
           1       0.29      0.67      0.40         3

    accuracy                           0.71        21
   macro avg       0.61      0.69      0.61        21
weighted avg       0.84      0.71      0.75        21

[[13  5]
 [ 1  2]]


In [71]:
y_pred_rf_04 = (y_prob_rf >= 0.4).astype(int)

print("Random Forest Tuned - threshold 0.4")
print(classification_report(y_test, y_pred_rf_04))
print(confusion_matrix(y_test, y_pred_rf_04))

Random Forest Tuned - threshold 0.4
              precision    recall  f1-score   support

           0       0.94      0.89      0.91        18
           1       0.50      0.67      0.57         3

    accuracy                           0.86        21
   macro avg       0.72      0.78      0.74        21
weighted avg       0.88      0.86      0.87        21

[[16  2]
 [ 1  2]]


In [72]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train_smote, y_train_smote)

y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_pred_xgb = (y_prob_xgb >= 0.5).astype(int)

print("XGBoost - threshold 0.5")
print(classification_report(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

XGBoost - threshold 0.5
              precision    recall  f1-score   support

           0       0.94      0.83      0.88        18
           1       0.40      0.67      0.50         3

    accuracy                           0.81        21
   macro avg       0.67      0.75      0.69        21
weighted avg       0.86      0.81      0.83        21

[[15  3]
 [ 1  2]]
ROC-AUC: 0.8518518518518519


In [73]:
y_pred_xgb_03 = (y_prob_xgb >= 0.3).astype(int)

print("XGBoost - threshold 0.3")
print(classification_report(y_test, y_pred_xgb_03))
print(confusion_matrix(y_test, y_pred_xgb_03))

XGBoost - threshold 0.3
              precision    recall  f1-score   support

           0       0.93      0.78      0.85        18
           1       0.33      0.67      0.44         3

    accuracy                           0.76        21
   macro avg       0.63      0.72      0.65        21
weighted avg       0.85      0.76      0.79        21

[[14  4]
 [ 1  2]]


In [74]:
y_pred_xgb_04 = (y_prob_xgb >= 0.4).astype(int)

print("XGBoost - threshold 0.4")
print(classification_report(y_test, y_pred_xgb_04))
print(confusion_matrix(y_test, y_pred_xgb_04))

XGBoost - threshold 0.4
              precision    recall  f1-score   support

           0       0.94      0.83      0.88        18
           1       0.40      0.67      0.50         3

    accuracy                           0.81        21
   macro avg       0.67      0.75      0.69        21
weighted avg       0.86      0.81      0.83        21

[[15  3]
 [ 1  2]]


In [75]:
import joblib

joblib.dump(xgb_model, "final_sepsis_model.pkl")

['final_sepsis_model.pkl']

In [76]:
results.to_csv("model_results.csv", index=False)
from google.colab import files
files.download("model_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [78]:
from sklearn.metrics import classification_report

report = classification_report(y_test, y_pred_xgb_04)

with open("classification_report.txt", "w") as f:
    f.write(report)

from google.colab import files
files.download("classification_report.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [79]:
from google.colab import files
files.download("final_sepsis_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [80]:
import joblib
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Save final model
joblib.dump(xgb_model, "final_sepsis_model.pkl")

# 2. Save patient-level dataset
patient_df.to_csv("patient_level_data.csv", index=False)

# 3. Save predictions
predictions_df = X_test.copy()
predictions_df["Actual"] = y_test.values
predictions_df["Predicted"] = y_pred_xgb_04
predictions_df["Probability"] = y_prob_xgb

predictions_df.to_csv("predictions.csv", index=False)

# 4. Save classification report
report = classification_report(y_test, y_pred_xgb_04)

with open("classification_report.txt", "w") as f:
    f.write(report)

# 5. Save confusion matrix
cm = confusion_matrix(y_test, y_pred_xgb_04)

cm_df = pd.DataFrame(cm, columns=["Pred_0", "Pred_1"], index=["Actual_0", "Actual_1"])
cm_df.to_csv("confusion_matrix.csv")

# 6. Save ROC-AUC score
roc_auc = roc_auc_score(y_test, y_prob_xgb)

with open("roc_auc.txt", "w") as f:
    f.write(f"ROC-AUC Score: {roc_auc}")

print("✅ All files saved successfully!")

✅ All files saved successfully!
